# Qualitative Analysis of Best and Worst LLM Priors

This notebook analyzes the best and worst performing LLM priors for each model across three domains (Glassdoor, NHANES, PitchBook) based on:
- **Error Ratio**: Relative error (abs_error / ground_truth)
- **ECE**: Expected Calibration Error (using std_ratio as proxy)
- **CRPS**: Continuous Ranked Probability Score

For each best/worst case, we display:
1. Variable information (description, conditions, ground truth)
2. Fitted prior parameters
3. Full reasoning chain from the LLM

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from IPython.display import display, Markdown, HTML
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# Paths
EXPERIMENTS_DIR = Path('/home/ubuntu/openestimate/experiments')
DATA_DIR = Path('/home/ubuntu/openestimate/data')

# Domains to analyze
DOMAINS = ['glassdoor', 'nhanes', 'pitchbook']

# Metrics to analyze
METRICS = {
    'error_ratio': 'Error Ratio',
    'crps': 'CRPS',
    'std_ratio': 'Std Ratio (Calibration Proxy)'
}

# Number of top/bottom examples to show per model per domain
N_EXAMPLES = 1

## Helper Functions

In [ ]:
def load_domain_data(domain):
    """Load combined results and variable info for a domain."""
    results_path = EXPERIMENTS_DIR / domain / f'{domain}_combined_processed_results.csv'
    variables_path = DATA_DIR / 'variables' / f'{domain}_variables.json'
    results_with_posteriors_path = EXPERIMENTS_DIR / domain / 'results_with_posteriors.csv'
    
    results = pd.read_csv(results_path)
    with open(variables_path) as f:
        variables = json.load(f)
    
    # Load results with posteriors which includes statistical baselines
    baselines_df = pd.DataFrame()
    if results_with_posteriors_path.exists():
        all_results = pd.read_csv(results_with_posteriors_path)
        # Filter for n=5 statistical baselines
        baselines_df = all_results[all_results['approach'].str.contains('statistical_baseline_n5', na=False)]
    
    return results, variables, baselines_df


def load_elicitation_conversation(convo_file_path, variable_name):
    """Load the elicitation conversation for a specific variable."""
    if pd.isna(convo_file_path):
        return None
    
    # Expand home directory
    convo_file_path = Path(convo_file_path.replace('~', '/home/ubuntu'))
    
    if not convo_file_path.exists():
        # Try to find it in elicitation_cache
        cache_dir = convo_file_path.parent / 'elicitation_cache'
        if cache_dir.exists():
            # Try to find by variable short name
            cache_files = list(cache_dir.glob('*_prior.json'))
            for cache_file in cache_files:
                with open(cache_file) as f:
                    data = json.load(f)
                    if variable_name in data:
                        # Handle nested structure
                        if variable_name in data[variable_name]:
                            return data[variable_name][variable_name]
                        return data[variable_name]
        return None
    
    # Load from main elicited_priors.json
    try:
        with open(convo_file_path) as f:
            data = json.load(f)
            if variable_name in data:
                # Handle nested structure: data[variable_name][variable_name]
                if variable_name in data[variable_name]:
                    return data[variable_name][variable_name]
                return data[variable_name]
    except Exception as e:
        print(f"Error loading {convo_file_path}: {e}")
        return None
    
    return None


def extract_reasoning(elicitation_data):
    """Extract the reasoning chain from elicitation data."""
    if elicitation_data is None:
        return "Reasoning not available"
    
    if 'conversation' in elicitation_data:
        # Find assistant's response
        for turn in elicitation_data['conversation']:
            if turn.get('role') == 'assistant':
                return turn.get('content', 'No content')
    
    return "Reasoning not found in conversation"


def format_prior_params(row):
    """Format prior parameters based on distribution type."""
    dist_type = row['fitted_distribution_type']
    
    if dist_type == 'lognormal':
        return {
            'type': 'lognormal',
            'mu': row['mu'],
            'sigma': row['sigma'],
            'mean': row['mean'],
            'median': row['median']
        }
    elif dist_type in ['normal', 'gaussian']:
        return {
            'type': dist_type,
            'mean': row['mean'],
            'std': row['std'],
            'median': row['median']
        }
    elif dist_type == 'beta':
        return {
            'type': 'beta',
            'a': row['a'],
            'b': row['b'],
            'mean': row['mean'],
            'median': row['median']
        }
    else:
        return {'type': dist_type}


def get_variable_info(variable_short_name, variables_dict):
    """Get variable information from variables dictionary."""
    if variable_short_name in variables_dict:
        var_info = variables_dict[variable_short_name]
        return {
            'full_description': var_info.get('variable', 'N/A'),
            'ground_truth_mean': var_info.get('mean', 'N/A'),
            'ground_truth_std': var_info.get('std', 'N/A'),
            'ground_truth_dist': var_info.get('ground_truth_distribution_type', 'N/A'),
            'conditions': var_info.get('nat_langs', []),
            'difficulty': var_info.get('difficulty', 'base')
        }
    return None


def get_baseline_info(variable_short_name, baselines_df, llm_dist_type):
    """Get baseline n=5 information for a variable from the baselines dataframe.

    IMPORTANT: The error_ratio is computed by comparing to the baseline with the SAME
    distribution type as the LLM chose. See utils.py lines 424-431 and 448:
        - Groups by (variable, approach) 
        - Computes MEAN of abs_error and std across ALL trials
        - baseline_approach = f'statistical_baseline_n5_{fitted_dist_type}'

    So we need to:
    1. Filter for this variable
    2. Filter for matching baseline approach (includes distribution type)
    3. Take MEAN across all trials (not individual trial values)

    This matches the logic in utils.py:compute_error_ratios_and_std_ratios()
    """
    if baselines_df.empty:
        return None

    # Normalize distribution type names
    if llm_dist_type == 'normal':
        llm_dist_type = 'gaussian'

    # Build the expected baseline approach name
    baseline_approach = f'statistical_baseline_n5_{llm_dist_type}'

    # Filter for this variable's n=5 baselines with matching approach
    # The approach column will be like "statistical_baseline_n5_gaussian" or "statistical_baseline_n5_lognormal"
    var_baselines = baselines_df[
        (baselines_df['variable'] == variable_short_name) &
        (baselines_df['approach'] == baseline_approach)
    ]

    if var_baselines.empty:
        return None

    # Compute MEAN across all trials - this matches utils.py line 424-429
    # baseline_stats = results.groupby(['variable', 'approach']).agg({'abs_error': 'mean', 'std': 'mean'})
    avg_abs_error = var_baselines['abs_error'].mean()
    avg_crps = var_baselines['crps'].mean()
    avg_std = var_baselines['std'].mean()
    num_trials = len(var_baselines)

    return {
        'n5_abs_error': avg_abs_error,
        'n5_crps': avg_crps,
        'n5_std': avg_std,
        'n5_num_trials': num_trials,
        'baseline_approach': baseline_approach,
        'baseline_dist_type': llm_dist_type
    }


def display_example(row, variables_dict, baselines_df, elicitation_data, metric_name, rank_type):
    """Display a single example with all relevant information."""
    
    # Header
    display(Markdown(f"### {rank_type} Example - {metric_name}"))
    
    # Variable info
    var_info = get_variable_info(row['variable'], variables_dict)
    
    # Get LLM's fitted distribution type
    llm_dist_type = row.get('fitted_distribution_type', 'gaussian')
    baseline_info = get_baseline_info(row['variable'], baselines_df, llm_dist_type)
    
    display(Markdown("#### Variable Information"))
    info_html = f"""
    <div style='background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin: 10px 0; color: #000000;'>
        <p style='color: #000000;'><strong>Variable ID:</strong> {row['variable']}</p>
        <p style='color: #000000;'><strong>Description:</strong> {row['variable_name']}</p>
    """
    
    if var_info:
        info_html += f"""
        <p style='color: #000000;'><strong>Difficulty:</strong> {var_info['difficulty']}</p>
        <p style='color: #000000;'><strong>Conditions:</strong> {', '.join(var_info['conditions']) if var_info['conditions'] else 'None (base variable)'}</p>
        <p style='color: #000000;'><strong>Ground Truth Mean:</strong> {var_info['ground_truth_mean']:.2f}</p>
        <p style='color: #000000;'><strong>Ground Truth Std:</strong> {var_info['ground_truth_std']:.2f}</p>
        <p style='color: #000000;'><strong>Ground Truth Distribution:</strong> {var_info['ground_truth_dist']}</p>
        """
    
    info_html += "</div>"
    display(HTML(info_html))
    
    # Prior parameters
    display(Markdown("#### Fitted Prior"))
    prior_params = format_prior_params(row)
    prior_html = f"""
    <div style='background-color: #e3f2fd; padding: 15px; border-radius: 5px; margin: 10px 0; color: #000000;'>
        <p style='color: #000000;'><strong>Distribution Type:</strong> {prior_params['type']}</p>
    """
    
    for key, value in prior_params.items():
        if key != 'type' and pd.notna(value):
            if isinstance(value, (int, float)):
                prior_html += f"<p style='color: #000000;'><strong>{key}:</strong> {value:.4f}</p>"
            else:
                prior_html += f"<p style='color: #000000;'><strong>{key}:</strong> {value}</p>"
    
    prior_html += "</div>"
    display(HTML(prior_html))
    
    # Metrics with baseline comparison
    display(Markdown("#### Performance Metrics"))
    metrics_html = f"""
    <div style='background-color: #fff3e0; padding: 15px; border-radius: 5px; margin: 10px 0; color: #000000;'>
        <p style='color: #000000;'><strong>LLM Absolute Error:</strong> {row['abs_error']:.4f}</p>
        <p style='color: #000000;'><strong>LLM MAE (Mean):</strong> {row['mae_mean']:.4f}</p>
        <p style='color: #000000;'><strong>LLM MAE (Median):</strong> {row['mae_median']:.4f}</p>
        <p style='color: #000000;'><strong>LLM CRPS:</strong> {row['crps']:.4f}</p>
        <p style='color: #000000;'><strong>LLM Std Ratio:</strong> {row['std_ratio']:.4f}</p>
    """
    
    if baseline_info:
        metrics_html += f"""
        <hr style='margin: 10px 0; border: 1px solid #ccc;'>
        <p style='color: #000000; font-weight: bold; margin-top: 10px;'>N=5 Baseline (approach: {baseline_info['baseline_approach']}):</p>
        <p style='color: #000000; font-style: italic;'>Mean computed across {baseline_info['n5_num_trials']} trials (matching utils.py:424-429)</p>
        <p style='color: #000000;'><strong>Baseline Absolute Error (mean across trials):</strong> {baseline_info['n5_abs_error']:.4f}</p>
        <p style='color: #000000;'><strong>Baseline CRPS (mean across trials):</strong> {baseline_info['n5_crps']:.4f}</p>
        <p style='color: #000000;'><strong>Baseline Std (mean across trials):</strong> {baseline_info['n5_std']:.4f}</p>
        """
        
        # Validation test: LLM error ratio should equal LLM abs_error / Baseline abs_error (matching dist type)
        computed_error_ratio = row['abs_error'] / baseline_info['n5_abs_error'] if baseline_info['n5_abs_error'] != 0 else np.nan
        
        # Also compute std_ratio for validation
        llm_std = row.get('std', np.nan)
        computed_std_ratio = llm_std / baseline_info['n5_std'] if baseline_info['n5_std'] != 0 else np.nan
        
        metrics_html += f"""
        <hr style='margin: 10px 0; border: 1px dashed #999;'>
        <p style='color: #000000; font-weight: bold; margin-top: 10px;'>Validation Tests:</p>
        <p style='color: #000000;'><strong>Error Ratio (from CSV):</strong> {row['error_ratio']:.6f}</p>
        <p style='color: #000000;'><strong>Error Ratio (computed):</strong> {computed_error_ratio:.6f}</p>
        <p style='color: #000000;'><strong>Match:</strong> {'✓ PASS' if pd.notna(computed_error_ratio) and abs(row['error_ratio'] - computed_error_ratio) < 0.01 else '✗ FAIL'}</p>
        <p style='color: #000000; margin-top: 5px;'><strong>Std Ratio (from CSV):</strong> {row['std_ratio']:.6f}</p>
        <p style='color: #000000;'><strong>Std Ratio (computed):</strong> {computed_std_ratio:.6f}</p>
        <p style='color: #000000;'><strong>Match:</strong> {'✓ PASS' if pd.notna(computed_std_ratio) and abs(row['std_ratio'] - computed_std_ratio) < 0.01 else '✗ FAIL'}</p>
        <p style='color: #000000; margin-top: 10px;'><em>Note: Error ratio < 1.0 means LLM is better than baseline</em></p>
        <p style='color: #000000;'><em>Std ratio ~ 1.0 means well-calibrated uncertainty</em></p>
        """
    
    metrics_html += "</div>"
    display(HTML(metrics_html))
    
    # Reasoning chain
    display(Markdown("#### LLM Reasoning Chain"))
    reasoning = extract_reasoning(elicitation_data)
    
    reasoning_html = f"""
    <div style='background-color: #f1f8e9; padding: 15px; border-radius: 5px; margin: 10px 0; 
                white-space: pre-wrap; font-family: monospace; max-height: 500px; overflow-y: auto; color: #000000;'>
    {reasoning}
    </div>
    """
    display(HTML(reasoning_html))
    
    display(Markdown("---"))

## Main Analysis

In [4]:
def analyze_domain(domain, metric, n_examples=3):
    """
    Analyze best and worst priors for a domain and metric.
    
    Args:
        domain: Domain name (glassdoor, nhanes, pitchbook)
        metric: Metric column name (error_ratio, crps, std_ratio)
        n_examples: Number of examples to show for best/worst
    """
    display(Markdown(f"# Analysis: {domain.upper()} - {METRICS.get(metric, metric)}"))
    
    # Load data
    results, variables, baselines = load_domain_data(domain)
    
    # Clean data - remove rows with NaN in the metric
    results_clean = results.dropna(subset=[metric]).copy()
    
    # For std_ratio, we want values close to 1.0 (well-calibrated)
    if metric == 'std_ratio':
        results_clean['metric_deviation'] = np.abs(results_clean[metric] - 1.0)
        sort_metric = 'metric_deviation'
        ascending_best = True
        ascending_worst = False
    else:
        sort_metric = metric
        ascending_best = True  # Lower is better for error_ratio and crps
        ascending_worst = False
    
    # Get unique models
    models = results_clean['model'].unique()
    
    display(Markdown(f"**Found {len(models)} models**: {', '.join(sorted(models))}"))
    display(Markdown(f"**Total rows**: {len(results_clean)}"))
    
    # Analyze each model
    for model in sorted(models):
        display(Markdown(f"## Model: {model}"))
        
        model_data = results_clean[results_clean['model'] == model].copy()
        
        if len(model_data) == 0:
            display(Markdown("*No data for this model*"))
            continue
        
        # Get best examples
        best_examples = model_data.nsmallest(n_examples, sort_metric) if ascending_best else model_data.nlargest(n_examples, sort_metric)
        
        # Get worst examples
        worst_examples = model_data.nlargest(n_examples, sort_metric) if ascending_best else model_data.nsmallest(n_examples, sort_metric)
        
        # Display best examples
        display(Markdown(f"### ✅ Best {n_examples} Examples (Lowest {METRICS.get(metric, metric)})"))
        
        for idx, (_, row) in enumerate(best_examples.iterrows(), 1):
            display(Markdown(f"#### Example {idx}/{n_examples}"))
            
            # Load elicitation data
            elicitation_data = load_elicitation_conversation(row['convo_file_path'], row['variable_name'])
            
            # Display
            display_example(row, variables, baselines, elicitation_data, METRICS.get(metric, metric), "BEST")
        
        # Display worst examples
        display(Markdown(f"### ❌ Worst {n_examples} Examples (Highest {METRICS.get(metric, metric)})"))
        
        for idx, (_, row) in enumerate(worst_examples.iterrows(), 1):
            display(Markdown(f"#### Example {idx}/{n_examples}"))
            
            # Load elicitation data
            elicitation_data = load_elicitation_conversation(row['convo_file_path'], row['variable_name'])
            
            # Display
            display_example(row, variables, baselines, elicitation_data, METRICS.get(metric, metric), "WORST")
    
    display(Markdown("\n\n"))

## Run Analysis for Each Domain and Metric

Choose which analyses to run below. You can comment out domains or metrics you don't want to analyze.

In [ ]:
# Configuration for which analyses to run
METRICS_TO_ANALYZE = ['error_ratio', 'crps', 'std_ratio']  # Comment out metrics you don't want

# Number of best/worst examples to show
N_EXAMPLES = 1

### Glassdoor Analysis

In [ ]:
# Run Glassdoor analysis
for metric in METRICS_TO_ANALYZE:
    analyze_domain('glassdoor', metric, n_examples=N_EXAMPLES)

### NHANES Analysis

In [ ]:
# Run NHANES analysis
for metric in METRICS_TO_ANALYZE:
    analyze_domain('nhanes', metric, n_examples=N_EXAMPLES)

### PitchBook Analysis

In [ ]:
# Run PitchBook analysis
for metric in METRICS_TO_ANALYZE:
    analyze_domain('pitchbook', metric, n_examples=N_EXAMPLES)

## Quick Summary Statistics

Get an overview of model performance across domains.

In [ ]:
def get_summary_stats(domain):
    """Get summary statistics for a domain."""
    results, _, _ = load_domain_data(domain)
    
    summary = results.groupby('model').agg({
        'error_ratio': ['mean', 'median', 'std', 'min', 'max'],
        'crps': ['mean', 'median', 'std', 'min', 'max'],
        'std_ratio': ['mean', 'median', 'std', 'min', 'max']
    }).round(4)
    
    return summary

### Glassdoor Summary Statistics

In [ ]:
display(Markdown(f"## Summary Statistics: GLASSDOOR"))
summary = get_summary_stats('glassdoor')
display(summary)

### NHANES Summary Statistics

In [ ]:
display(Markdown(f"## Summary Statistics: NHANES"))
summary = get_summary_stats('nhanes')
display(summary)

### PitchBook Summary Statistics

In [ ]:
display(Markdown(f"## Summary Statistics: PITCHBOOK"))
summary = get_summary_stats('pitchbook')
display(summary)

# Example: Analyze a specific model across all domains
specific_model = 'gpt-4o_base_unified_temp0.5'  # Change this to any model name

results, variables, baselines = load_domain_data('glassdoor')
model_data = results[results['model'] == specific_model]

if len(model_data) > 0:
    display(Markdown(f"## GLASSDOOR - {specific_model}"))
    display(Markdown(f"**Number of variables:** {len(model_data)}"))
    
    stats = model_data[['error_ratio', 'crps', 'std_ratio']].describe()
    display(stats)

In [ ]:
results, variables, baselines = load_domain_data('nhanes')
model_data = results[results['model'] == specific_model]

if len(model_data) > 0:
    display(Markdown(f"## NHANES - {specific_model}"))
    display(Markdown(f"**Number of variables:** {len(model_data)}"))
    
    stats = model_data[['error_ratio', 'crps', 'std_ratio']].describe()
    display(stats)

In [ ]:
results, variables, baselines = load_domain_data('pitchbook')
model_data = results[results['model'] == specific_model]

if len(model_data) > 0:
    display(Markdown(f"## PITCHBOOK - {specific_model}"))
    display(Markdown(f"**Number of variables:** {len(model_data)}"))
    
    stats = model_data[['error_ratio', 'crps', 'std_ratio']].describe()
    display(stats)

In [ ]:
# Example: Analyze a specific model across all domains
specific_model = 'gpt-4o_base_unified_temp0.5'  # Change this to any model name

for domain in DOMAINS_TO_ANALYZE:
    results, variables, baselines = load_domain_data(domain)
    model_data = results[results['model'] == specific_model]
    
    if len(model_data) > 0:
        display(Markdown(f"## {domain.upper()} - {specific_model}"))
        display(Markdown(f"**Number of variables:** {len(model_data)}"))
        
        stats = model_data[['error_ratio', 'crps', 'std_ratio']].describe()
        display(stats)
        display(Markdown("\n"))

results, _, _ = load_domain_data('glassdoor')

display(Markdown(f"## Difficulty Analysis: GLASSDOOR"))

# Add difficulty column based on variable naming
def get_difficulty(var_name):
    if var_name.startswith('triple_'):
        return 'triple'
    elif var_name.startswith('double_'):
        return 'double'
    elif var_name.startswith('single_'):
        return 'single'
    else:
        return 'base'

results['difficulty_level'] = results['variable'].apply(get_difficulty)

difficulty_summary = results.groupby(['model', 'difficulty_level']).agg({
    'error_ratio': 'mean',
    'crps': 'mean',
    'std_ratio': 'mean'
}).round(4)

display(difficulty_summary)

### NHANES Difficulty Analysis

In [ ]:
results, _, _ = load_domain_data('nhanes')

display(Markdown(f"## Difficulty Analysis: NHANES"))

def get_difficulty(var_name):
    if var_name.startswith('triple_'):
        return 'triple'
    elif var_name.startswith('double_'):
        return 'double'
    elif var_name.startswith('single_'):
        return 'single'
    else:
        return 'base'

results['difficulty_level'] = results['variable'].apply(get_difficulty)

difficulty_summary = results.groupby(['model', 'difficulty_level']).agg({
    'error_ratio': 'mean',
    'crps': 'mean',
    'std_ratio': 'mean'
}).round(4)

display(difficulty_summary)

## Difficulty Analysis

Analyze performance by variable difficulty (single, double, triple conditions).

In [ ]:
for domain in DOMAINS_TO_ANALYZE:
    results, _, _ = load_domain_data(domain)
    
    display(Markdown(f"## Difficulty Analysis: {domain.upper()}"))
    
    # Add difficulty column based on variable naming
    def get_difficulty(var_name):
        if var_name.startswith('triple_'):
            return 'triple'
        elif var_name.startswith('double_'):
            return 'double'
        elif var_name.startswith('single_'):
            return 'single'
        else:
            return 'base'
    
    results['difficulty_level'] = results['variable'].apply(get_difficulty)
    
    difficulty_summary = results.groupby(['model', 'difficulty_level']).agg({
        'error_ratio': 'mean',
        'crps': 'mean',
        'std_ratio': 'mean'
    }).round(4)
    
    display(difficulty_summary)
    display(Markdown("\n"))